# Malaria Cell Images

Mục tiêu: **accuracy cao** nhưng **thời gian tốt hơn**.

Chiến lược:
- Train **2-stage**: `224px` (nhanh) → fine-tune `384px` (giữ/đẩy accuracy)
- Model: `convnext_tiny.in12k_ft_in1k`
- AMP + AdamW + OneCycleLR + EarlyStopping
- Evaluate cuối: **TTA bật sẵn**

Dataset: Kaggle `iarunava/cell-images-for-detecting-malaria`

In [ ]:
import os, sys, random, gc
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from tqdm.auto import tqdm

from torch.amp import autocast, GradScaler

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("timm:", timm.__version__)
print("Alb:", A.__version__)
print("OpenCV:", cv2.__version__)

Python: 3.12.12
Torch: 2.8.0+cu126
timm: 1.0.20
Alb: 2.0.8
OpenCV: 4.12.0


In [ ]:
# 1) Config
@dataclass
class CFG:
    SEED: int = 42
    NUM_WORKERS: int = 4
    PIN_MEMORY: bool = True

    IMG1: int = 224
    IMG2: int = 384
    BS1: int = 128
    BS2: int = 32       # if OOM -> 16
    EPOCHS1: int = 5
    EPOCHS2: int = 4
    PATIENCE: int = 2

    LR1: float = 5e-4
    LR2: float = 2e-4
    WEIGHT_DECAY: float = 1e-4
    LABEL_SMOOTHING: float = 0.02

    MODEL: str = "convnext_tiny.in12k_ft_in1k"

cfg = CFG()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def clean_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

set_seed(cfg.SEED)

Device: cuda
GPU: Tesla T4


## 2) Locate dataset path on Kaggle

Sau khi Add dataset vào notebook, path thường là:
- `/kaggle/input/cell-images-for-detecting-malaria/cell_images/Parasitized`
- `/kaggle/input/cell-images-for-detecting-malaria/cell_images/Uninfected`

In [ ]:
import os, glob
from pathlib import Path

print("Datasets in /kaggle/input:")
print(os.listdir("/kaggle/input"))

paras_candidates = glob.glob("/kaggle/input/**/Parasitized", recursive=True)
uninf_candidates = glob.glob("/kaggle/input/**/Uninfected", recursive=True)

print("Found Parasitized:", paras_candidates[:3])
print("Found Uninfected:", uninf_candidates[:3])

assert len(paras_candidates) > 0 and len(uninf_candidates) > 0, \
    "Bạn chưa Add dataset hoặc dataset không đúng cấu trúc."

PARA_DIR = Path(paras_candidates[0])
UNINF_DIR = Path(uninf_candidates[0])
DATA_DIR = PARA_DIR.parent

print("✅ DATA_DIR:", DATA_DIR)
print("✅ PARA_DIR:", PARA_DIR)
print("✅ UNINF_DIR:", UNINF_DIR)
print("Parasitized count:", len(list(PARA_DIR.glob('*'))))
print("Uninfected count:", len(list(UNINF_DIR.glob('*'))))


Datasets in /kaggle/input:
['cell-images-for-detecting-malaria']
Found Parasitized: ['/kaggle/input/cell-images-for-detecting-malaria/cell_images/Parasitized', '/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images/Parasitized']
Found Uninfected: ['/kaggle/input/cell-images-for-detecting-malaria/cell_images/Uninfected', '/kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images/Uninfected']
✅ DATA_DIR: /kaggle/input/cell-images-for-detecting-malaria/cell_images
✅ PARA_DIR: /kaggle/input/cell-images-for-detecting-malaria/cell_images/Parasitized
✅ UNINF_DIR: /kaggle/input/cell-images-for-detecting-malaria/cell_images/Uninfected
Parasitized count: 13780
Uninfected count: 13780


In [ ]:
# 3) Build df + stratified split 80/20
paras_files = sorted([str(p) for p in PARA_DIR.glob("*") if p.suffix.lower() in [".png",".jpg",".jpeg"]])
uninf_files = sorted([str(p) for p in UNINF_DIR.glob("*") if p.suffix.lower() in [".png",".jpg",".jpeg"]])

df = pd.DataFrame({"path": paras_files + uninf_files,
                   "label": [1]*len(paras_files) + [0]*len(uninf_files)})

train_df, val_df = train_test_split(df, test_size=0.2, random_state=cfg.SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Total:", len(df), "| Train:", len(train_df), "| Val:", len(val_df))
print("Train dist:", train_df["label"].value_counts(normalize=True).to_dict())
print("Val dist:", val_df["label"].value_counts(normalize=True).to_dict())

Total: 27558 | Train: 22046 | Val: 5512
Train dist: {0: 0.5, 1: 0.5}
Val dist: {1: 0.5, 0: 0.5}


In [ ]:
# 4) Transforms
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def get_train_tfms(img_size):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.80, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=15, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=0, border_mode=cv2.BORDER_REFLECT_101, p=0.4),
        A.RandomBrightnessContrast(0.12, 0.12, p=0.5),
        A.GaussNoise(var_limit=(3.0, 10.0), p=0.10),
        A.CoarseDropout(max_holes=4, max_height=24, max_width=24, min_holes=1, p=0.10),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_tfms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def read_rgb(path):
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

In [ ]:
# 5) Dataset + DataLoader factory
class MalariaDS(Dataset):
    def __init__(self, df, tfms):
        self.df = df.reset_index(drop=True)
        self.tfms = tfms
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        y = int(self.df.loc[idx, "label"])
        img = read_rgb(path)
        x = self.tfms(image=img)["image"]
        return x, torch.tensor(y, dtype=torch.long)

def make_loaders(img_size, batch_size):
    train_tfms = get_train_tfms(img_size)
    val_tfms = get_val_tfms(img_size)
    train_ds = MalariaDS(train_df, train_tfms)
    val_ds = MalariaDS(val_df, val_tfms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                              persistent_workers=(cfg.NUM_WORKERS>0),
                              prefetch_factor=4, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY,
                            persistent_workers=(cfg.NUM_WORKERS>0),
                            prefetch_factor=4)
    return train_loader, val_loader, val_ds

In [ ]:
# 6) Model + train/eval loops
def build_model():
    m = timm.create_model(cfg.MODEL, pretrained=True, num_classes=2, drop_rate=0.2)
    try:
        m.set_grad_checkpointing(True)
    except Exception:
        pass
    return m

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    losses, preds_all, y_all = [], [], []
    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        losses.append(loss.item())
        preds_all.append(logits.argmax(1).detach().cpu().numpy())
        y_all.append(y.detach().cpu().numpy())
    y_true = np.concatenate(y_all)
    y_pred = np.concatenate(preds_all)
    return float(np.mean(losses)), float(accuracy_score(y_true, y_pred)), y_true, y_pred

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    losses, accs = [], []
    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(loss.item())
        accs.append((logits.argmax(1) == y).float().mean().item())
    return float(np.mean(losses)), float(np.mean(accs))

In [ ]:
# 7) 2-stage training
BEST_PATH = Path("/kaggle/working/best_convnext_224to384.pth")

model = build_model().to(device)
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
scaler = GradScaler("cuda", enabled=torch.cuda.is_available())

def run_stage(stage_name, img_size, batch_size, epochs, lr):
    print(f"\n==== {stage_name} | IMG={img_size} BS={batch_size} epochs={epochs} lr={lr} ====")
    train_loader, val_loader, val_ds = make_loaders(img_size, batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=len(train_loader),
        pct_start=0.2, div_factor=10.0, final_div_factor=200.0
    )

    best_acc = -1.0
    pat = 0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler)
        va_loss, va_acc, y_true, y_pred = eval_one_epoch(model, val_loader, criterion)

        print(f"{stage_name} Epoch {epoch:02d}/{epochs} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} ({va_acc*100:.2f}%)")

        if va_acc > best_acc:
            best_acc = va_acc
            torch.save({"model": model.state_dict(), "best_acc": best_acc, "stage": stage_name, "img_size": img_size}, BEST_PATH)
            pat = 0
            print(f"  -> Saved BEST ({best_acc*100:.2f}%)")
        else:
            pat += 1
            if pat >= cfg.PATIENCE:
                print("  -> Early stopping this stage.")
                break

    clean_mem()
    return best_acc, val_ds

best1, _ = run_stage("STAGE1_224", cfg.IMG1, cfg.BS1, cfg.EPOCHS1, cfg.LR1)
best2, val_ds_384 = run_stage("STAGE2_384", cfg.IMG2, cfg.BS2, cfg.EPOCHS2, cfg.LR2)

print("\nBest checkpoint:", BEST_PATH)

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Using 2 GPUs

==== STAGE1_224 | IMG=224 BS=128 epochs=5 lr=0.0005 ====


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/2510045169.py:15: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(3.0, 10.0), p=0.10),
/tmp/ipykernel_55/2510045169.py:16: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=4, max_height=24, max_width=24, min_holes=1, p=0.10),


  0%|          | 0/172 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:192: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


...

  0%|          | 0/173 [00:00<?, ?it/s]

STAGE2_384 Epoch 03/4 | train loss 0.1441 acc 0.9637 | val loss 0.1247 acc 0.9722 (97.22%)
  -> Saved BEST (97.22%)


  0%|          | 0/688 [00:00<?, ?it/s]

  0%|          | 0/173 [00:00<?, ?it/s]

STAGE2_384 Epoch 04/4 | train loss 0.1270 acc 0.9708 | val loss 0.1214 acc 0.9742 (97.42%)
  -> Saved BEST (97.42%)

Best checkpoint: /kaggle/working/best_convnext_224to384.pth


In [ ]:
# 8) Final evaluation (NO TTA) @384
ckpt = torch.load(BEST_PATH, map_location=device)
best_model = build_model().to(device)
state_dict = ckpt["model"]

# nếu checkpoint train bằng DataParallel thì strip "module."
if any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

best_model.load_state_dict(state_dict, strict=True)


_, val_loader_384, val_ds_384 = make_loaders(cfg.IMG2, cfg.BS2)

va_loss, va_acc, y_true, y_pred = eval_one_epoch(best_model, val_loader_384, criterion)
print(f"FINAL (NO TTA) | val loss: {va_loss:.4f} | val acc: {va_acc*100:.2f}%")

print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nReport:\n", classification_report(y_true, y_pred, digits=4))

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/2510045169.py:15: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(3.0, 10.0), p=0.10),
/tmp/ipykernel_55/2510045169.py:16: UserWarning: Argument(s) 'max_holes, max_height, max_width, min_holes' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=4, max_height=24, max_width=24, min_holes=1, p=0.10),


  0%|          | 0/173 [00:00<?, ?it/s]

FINAL (NO TTA) | val loss: 0.1214 | val acc: 97.42%
Confusion matrix:
 [[2690   66]
 [  76 2680]]

Report:
               precision    recall  f1-score   support

           0     0.9725    0.9761    0.9743      2756
           1     0.9760    0.9724    0.9742      2756

    accuracy                         0.9742      5512
   macro avg     0.9742    0.9742    0.9742      5512
weighted avg     0.9742    0.9742    0.9742      5512



In [ ]:
# 9) Final evaluation WITH TTA @384
@torch.no_grad()
def eval_with_tta(model, ds, batch_size):
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY)
    tta_ops = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
        lambda x: torch.flip(x, dims=[2]),
        lambda x: torch.rot90(x, k=1, dims=[2,3]),
    ]
    all_pred, all_true = [], []
    for x, y in tqdm(loader, desc="TTA", leave=False):
        x = x.to(device, non_blocking=True)
        logits_sum = 0.0
        for op in tta_ops:
            xt = op(x)
            with autocast("cuda", enabled=torch.cuda.is_available()):
                logits_sum = logits_sum + model(xt)
        logits = logits_sum / len(tta_ops)
        all_pred.append(logits.argmax(1).detach().cpu().numpy())
        all_true.append(y.numpy())
    y_pred = np.concatenate(all_pred)
    y_true = np.concatenate(all_true)
    return float(accuracy_score(y_true, y_pred)), y_true, y_pred

tta_acc, yt, yp = eval_with_tta(best_model, val_ds_384, cfg.BS2)
print(f"FINAL (TTA) | val acc: {tta_acc*100:.2f}%")

print("TTA Confusion matrix:\n", confusion_matrix(yt, yp))
print("\nTTA Report:\n", classification_report(yt, yp, digits=4))

TTA:   0%|          | 0/173 [00:00<?, ?it/s]

FINAL (TTA) | val acc: 97.59%
TTA Confusion matrix:
 [[2694   62]
 [  71 2685]]

TTA Report:
               precision    recall  f1-score   support

           0     0.9743    0.9775    0.9759      2756
           1     0.9774    0.9742    0.9758      2756

    accuracy                         0.9759      5512
   macro avg     0.9759    0.9759    0.9759      5512
weighted avg     0.9759    0.9759    0.9759      5512



In [ ]:
print("======== FINAL SUMMARY ========")
print("Stage1: IMG", cfg.IMG1, "BS", cfg.BS1, "epochs", cfg.EPOCHS1)
print("Stage2: IMG", cfg.IMG2, "BS", cfg.BS2, "epochs", cfg.EPOCHS2)
print("Checkpoint:", BEST_PATH)

======== FINAL SUMMARY ========
Stage1: IMG 224 BS 128 epochs 5
Stage2: IMG 384 BS 32 epochs 4
Checkpoint: /kaggle/working/best_convnext_224to384.pth
